In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
import os
import sys

sys.path.append(str(Path.cwd().parent))
from src.database import SalesDatabase

In [ ]:
#Configurar db

PROCESSED_DIR = Path.cwd().parent / 'data' / 'processed'
db = SalesDatabase()
db.connect()

print(f"Diretorio dos dados processados: {PROCESSED_DIR}")
print(f"Banco: {db.db_path}")

In [ ]:
#Carregamento dos dados limpos

def carregar_dados_limpos():
    dados = {}
    arquivos = ['orders', 'customers', 'products', 'order_items', 
                'orders_payments', 'order_reviews', 'sellers']

    for nome in arquivos:
        caminho = PROCESSED_DIR / f'{nome}_clean.csv'
        if caminho.exists():
            dados[nome] = pd.read_csv(caminho)
            print(f"{nome}: {len(dados[nome])} registros")
        else:
            print(f"{nome}_clean.csv não encontrado")

    return dados

dados_limpos = carregar_dados_limpos()

In [ ]:
#Criação das tabelas SQL

for table_name, df in dados_limpos.items():
    df.to_sql(table_name, db.connection,
              if_exists='replace',
              index=False)
    print(f"Tabela {table_name} criada com {len(df)} registros.")

db.create_indexes()

In [ ]:
#Verificação dos bancos criados

print("Verificando Bancos de Dados...")
print("="*60)

#Listar tabelas
query_tables = """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
"""

tabelas = db.execute_query(query_tables)
print("Tabelas criadas:")
for tabela in tabelas['name']:
    count_query = f"SELECT COUNT(*) as total FROM {tabela}"

